# Tahap 3 CBR: Case Retrieval
### CBR Putusan Wanprestasi

**Tim:**
- Ahmad Qayyim — 202310370311286
- Bintang Mars Satria Tuhu — 202310370311410

---

## Tujuan

Membangun sistem retrieval yang dapat:
1. **TF-IDF + Cosine Similarity** sebagai baseline retrieval
2. **SVM dan Naive Bayes** untuk klasifikasi `kategori_solusi`
3. Generate 7 test queries (5 synthetic + 2 manual)
4. Save artifacts untuk Tahap 4 (Solution Reuse)

Split data: 80% train / 20% test, random_state=42 untuk reproducibility.

## 1. Setup

In [ ]:
!pip install scikit-learn pandas numpy joblib -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/kuliah/smt 6/Penalaran Komputer/Tugas SUB-CPMK 3/cbr-putusan-wanprestasi')
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## 2. Verifikasi Input dari Tahap 2

In [ ]:
cases_path = PROJECT_DIR / 'data' / 'processed' / 'cases.csv'
print(f'cases.csv exists: {cases_path.exists()}')

import pandas as pd
df = pd.read_csv(cases_path)
print(f'Shape : {df.shape}')
print(f'\nDistribusi kategori_solusi:')
print(df['kategori_solusi'].value_counts())

## 3. Run Retrieval Pipeline

In [ ]:
import sys
import importlib.util

sys.path.insert(0, str(PROJECT_DIR / 'src'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

retr_mod = load_module('retr_mod', PROJECT_DIR / 'src' / '04_retrieval.py')
result   = retr_mod.run()

retriever   = result['retriever']
classifiers = result['classifiers']
queries     = result['queries']
svm_metrics = result['svm_metrics']
nb_metrics  = result['nb_metrics']

## 4. Demo Retrieval — Test Function `retrieve()`

In [ ]:
# Coba dengan query manual sederhana
query_demo = (
    "Tergugat tidak membayar angsuran kredit pinjaman sesuai dengan "
    "perjanjian yang telah ditandatangani bersama penggugat."
)

print(f'Query: {query_demo}')
print('='*70)
results = retriever.retrieve(query_demo, k=5)
for r in results:
    print(f"\n[Rank {r['ranking']}]  Case: {r['case_id']}  |  Similarity: {r['similarity_score']:.4f}")
    print(f"  No Perkara : {r['no_perkara']}")
    print(f"  Kategori   : {r['kategori_solusi']}")
    print(f"  Pasal      : {r['pasal'][:80]}")
    print(f"  Amar       : {r['amar_putusan'][:150]}...")

## 5. Inspeksi Test Queries

In [ ]:
print(f'Total queries: {len(queries)}\n')
for q in queries:
    print(f"[{q['query_id']}] ({q['source']}) — expected: {q['expected_kategori']}")
    print(f"   Query     : {q['query_text'][:120]}...")
    print(f"   GT case_id: {q['ground_truth_case_ids']}")
    print()

## 6. Validasi Retrieval pada Synthetic Queries

Synthetic queries dibuat dari ringkasan_fakta case asli. Sistem retrieval seharusnya bisa rank case sumber-nya di top-K. Ini self-validation pertama.

In [ ]:
synthetic_queries = [q for q in queries if q['source'] == 'synthetic']

print('Self-validation (synthetic queries):')
print('='*70)
for q in synthetic_queries:
    results = retriever.retrieve(q['query_text'], k=5)
    retrieved_ids = [r['case_id'] for r in results]
    gt = q['ground_truth_case_ids'][0]
    
    if gt in retrieved_ids:
        rank = retrieved_ids.index(gt) + 1
        status = f'OK (rank #{rank})'
    else:
        status = 'MISSED (not in top-5)'
    
    print(f"{q['query_id']} ({q['expected_kategori']:25s}) GT={gt}  →  {status}")
    print(f"   Top-5: {retrieved_ids}")

## 7. Klasifikasi: SVM vs Naive Bayes

In [ ]:
print('Perbandingan Performa Classifier:')
print('='*70)
print(f'{"Metric":<25}  {"SVM":>10}  {"Naive Bayes":>15}')
print('-'*70)
for k in ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted',
          'precision_macro', 'recall_macro', 'f1_macro']:
    print(f'{k:<25}  {svm_metrics[k]:>10.4f}  {nb_metrics[k]:>15.4f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics_names = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']
svm_vals = [svm_metrics[m] for m in metrics_names]
nb_vals  = [nb_metrics[m]  for m in metrics_names]

x = np.arange(len(metrics_names))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, svm_vals, w, label='SVM', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + w/2, nb_vals,  w, label='Naive Bayes', color='coral', edgecolor='black')

ax.set_ylabel('Score')
ax.set_title('Perbandingan Performance: SVM vs Naive Bayes')
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in metrics_names])
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for b in bars:
        h = b.get_height()
        ax.annotate(f'{h:.3f}', xy=(b.get_x()+b.get_width()/2, h),
                    xytext=(0,3), textcoords='offset points',
                    ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('reports/figures/classifier_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix untuk model terbaik (yang accuracy lebih tinggi)
from sklearn.metrics import confusion_matrix
import seaborn as sns

best_name = 'SVM' if svm_metrics['accuracy'] >= nb_metrics['accuracy'] else 'Naive Bayes'
best_pred = classifiers['svm_pred'] if best_name == 'SVM' else classifiers['nb_pred']
y_test = classifiers['y_test']
labels  = sorted(set(list(y_test) + list(best_pred)))

cm = confusion_matrix(y_test, best_pred, labels=labels)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}  (Test set, n={len(y_test)})')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.xticks(rotation=20, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('reports/figures/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nBest classifier: {best_name}')

## 8. Output Files Verifikasi

In [ ]:
expected_files = [
    'data/eval/queries.json',
    'data/processed/tfidf_model.pkl',
    'data/processed/tfidf_matrix.pkl',
    'data/processed/svm_model.pkl',
    'data/processed/nb_model.pkl',
    'reports/figures/classifier_comparison.png',
    'reports/figures/confusion_matrix.png',
]
for f in expected_files:
    p = PROJECT_DIR / f
    size = p.stat().st_size / 1024 if p.exists() else 0
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}]  {f}  ({size:.1f} KB)')

## 9. Final Checklist

In [ ]:
from pathlib import Path

checks = {
    'TF-IDF model trained'       : retriever.vectorizer is not None,
    'TF-IDF matrix built'         : retriever.tfidf_matrix is not None,
    'SVM trained'                : classifiers['svm'] is not None,
    'Naive Bayes trained'         : classifiers['nb'] is not None,
    f'Queries generated (≥5)'    : len(queries) >= 5,
    'queries.json saved'         : Path('data/eval/queries.json').exists(),
    'Model artifacts saved (4)'  : all(Path(f'data/processed/{f}').exists() for f in
                                       ['tfidf_model.pkl', 'tfidf_matrix.pkl', 'svm_model.pkl', 'nb_model.pkl']),
}

for k, v in checks.items():
    print(f'  [{"OK" if v else "FAIL"}]  {k}')

if all(checks.values()):
    print('\nSTATUS: SIAP lanjut ke Tahap 4 (Solution Reuse)')
else:
    print('\nSTATUS: Ada checklist yang gagal, cek output di atas.')